# 第7课：完整反向传播训练

**学习目标：**
- 理解反向传播的完整流程
- 实现权重的逐层更新
- 看到网络在训练后分类效果的改善

---

前6课我们实现了前向传播、损失函数和需求函数。本课将它们组合成完整的训练循环，让网络真正"学会"分类。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import sys
sys.path.insert(0, '..')
from numpy.utils import create_data, plot_data

## 7.1 完整的网络实现

将前面所有组件整合：Layer、Network、激活函数、损失函数、需求函数。

In [ ]:
# ---- 激活函数 ----
def activation_ReLU(x):
    return np.maximum(0, x)

def activation_softmax(inputs):
    max_val = np.max(inputs, axis=1, keepdims=True)
    shifted = inputs - max_val
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals, axis=1, keepdims=True)

# ---- 损失与需求 ----
def one_hot(labels, num_classes=2):
    matrix = np.zeros((len(labels), num_classes))
    matrix[:, 1] = labels
    matrix[:, 0] = 1 - labels
    return matrix

def loss_function(predicted, real):
    real_matrix = one_hot(real)
    product = np.sum(predicted * real_matrix, axis=1)
    return 1 - product

def classify(probabilities):
    return np.rint(probabilities[:, 1]).astype(int)

def normalize(array):
    """按行标准化，防止梯度爆炸"""
    max_val = np.max(np.abs(array), axis=1, keepdims=True)
    scale = np.where(max_val == 0, 1, 1 / max_val)
    return array * scale

def vector_normalize(array):
    """向量标准化"""
    max_val = np.max(np.abs(array))
    scale = np.where(max_val == 0, 1, 1 / max_val)
    return array * scale

In [ ]:
class Layer:
    def __init__(self, n_inputs, n_neurons):
        self.weights = np.random.randn(n_inputs, n_neurons)
        self.biases = np.random.randn(n_neurons)

    def forward(self, inputs):
        self.sum = np.dot(inputs, self.weights) + self.biases
        return self.sum

    def backward(self, pre_layer_output, demands, learning_rate):
        """反向传播：计算梯度并更新权重
        
        参数:
            pre_layer_output: 前一层输出
            demands: 后一层传来的需求信号
            learning_rate: 学习率
        返回:
            传给前一层的需求信号
        """
        # 权重梯度
        weight_grad = np.dot(pre_layer_output.T, demands) / len(demands)
        # 更新权重
        self.weights += learning_rate * normalize(weight_grad)
        # 更新偏置
        bias_grad = np.mean(demands, axis=0)
        self.biases += learning_rate * vector_normalize(bias_grad)
        # 传播需求到前一层
        pre_demands = np.dot(demands, self.weights.T)
        # ReLU 导数
        pre_demands *= (pre_layer_output > 0).astype(float)
        return normalize(pre_demands)

In [ ]:
class Network:
    def __init__(self, network_shape, learning_rate=0.01):
        self.shape = network_shape
        self.learning_rate = learning_rate
        self.layers = []
        for i in range(len(network_shape) - 1):
            self.layers.append(Layer(network_shape[i], network_shape[i + 1]))

    def forward(self, inputs):
        """前向传播，保存每层输出"""
        self.layer_outputs = [inputs]
        for i in range(len(self.layers)):
            z = self.layers[i].forward(self.layer_outputs[i])
            if i == len(self.layers) - 1:
                a = activation_softmax(z)
            else:
                a = activation_ReLU(z)
                a = normalize(a)
            self.layer_outputs.append(a)
        return self.layer_outputs[-1]

    def get_demands(self, predicted, real):
        """计算输出层需求信号"""
        target = one_hot(real)
        for i in range(len(real)):
            if np.dot(target[i], predicted[i]) > 0.5:
                target[i] = np.array([0.0, 0.0])
            else:
                target[i] = (target[i] - 0.5) * 2
        return target

    def backward(self, real):
        """反向传播更新所有层"""
        demands = self.get_demands(self.layer_outputs[-1], real)
        for i in range(len(self.layers) - 1, -1, -1):
            demands = self.layers[i].backward(
                self.layer_outputs[i], demands, self.learning_rate
            )

## 7.2 训练循环

训练的核心流程：
1. **前向传播**：输入 → 预测
2. **计算损失**：预测 vs 真实
3. **反向传播**：计算梯度，更新权重
4. **重复**多个 epoch

In [ ]:
# 创建网络
net = Network([2, 16, 16, 2], learning_rate=0.05)

# 生成训练数据
train_data = create_data(500)
train_inputs = train_data[:, :2]
train_labels = train_data[:, 2].astype(int)

# 训练
n_epochs = 50
losses = []

for epoch in range(n_epochs):
    # 前向传播
    predicted = net.forward(train_inputs)
    # 计算损失
    loss = np.mean(loss_function(predicted, train_labels))
    losses.append(loss)
    # 反向传播
    net.backward(train_labels)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss:.4f}")

## 7.3 可视化训练过程

In [ ]:
plt.figure(figsize=(10, 4))

# Loss 曲线
plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("训练损失曲线")

# 分类结果
plt.subplot(1, 2, 2)
predicted = net.forward(train_inputs)
predictions = classify(predicted)
result_data = train_data.copy()
result_data[:, 2] = predictions
plot_data(result_data, "训练后的分类结果")

plt.tight_layout()
plt.show()

---

## 小结

- **训练循环**：前向 → 算损失 → 反向 → 更新权重 → 重复
- **反向传播**：从输出层逐层向前传播需求信号，更新每层权重
- **标准化**：防止梯度爆炸，保持训练稳定
- 训练后网络学会了区分环形边界

**下一课**我们将学习自动微分 — PyTorch 等框架的核心原理。